# 02 — Sky Coverage Maps (HEALPix)

This notebook is an **exploratory/diagnostic** step — it is not part of the
main prediction pipeline but provides a clear visual sanity check that our three
data sources (ZTF DR24, SDSS DR16Q, and the Stripe 82 footprint) overlap where
we expect them to.

**Inputs:**
- `data/matchfile_equcov_dr24_r.fits` — ZTF DR24 r-band epoch-count map
- `data/raw/DR16Q_v4.fits` — Full SDSS DR16Q quasar catalog

**Output:** Mollweide sky maps shown inline; no files written to disk.


In [ ]:
# Uncomment to install healpy if needed
# !pip install healpy


In [ ]:
import numpy as np
import healpy as hp
from astropy.io import fits
from astropy.wcs import WCS
import matplotlib.pyplot as plt

# ── HEALPix resolution ────────────────────────────────────────────────────────
# nside=256 gives ~0.23 deg pixels (~786,432 total pixels on the sky).
# Increase to 512 for higher resolution at the cost of more memory.
NSIDE = 256
NPIX  = hp.nside2npix(NSIDE)

# Convert HEALPix pixel centres to RA/Dec
theta, phi = hp.pix2ang(NSIDE, np.arange(NPIX))
ra_grid  = np.degrees(phi)
dec_grid = 90.0 - np.degrees(theta)


## 1. ZTF DR24 r-band Epoch-Count Map

We load the ZTF DR24 coverage FITS file, project it onto the HEALPix grid
via WCS, and display the number of observations per pixel.


In [ ]:
hdul    = fits.open("data/matchfile_equcov_dr24_r.fits")
ztf_data = hdul[0].data
ztf_wcs  = WCS(hdul[0].header)

# Map each HEALPix pixel centre to the corresponding ZTF FITS pixel
x_px, y_px = ztf_wcs.world_to_pixel_values(ra_grid, dec_grid)
x_px = np.floor(x_px).astype(int)
y_px = np.floor(y_px).astype(int)

# Build HEALPix map: copy epoch counts from ZTF FITS image
ztf_map = np.zeros(NPIX)
valid = (
    (x_px >= 0) & (x_px < ztf_data.shape[1]) &
    (y_px >= 0) & (y_px < ztf_data.shape[0])
)
ztf_map[valid] = ztf_data[y_px[valid], x_px[valid]]

hp.mollview(ztf_map, title="ZTF DR24 r-band — epoch counts per pixel")
hp.mollview((ztf_map > 0).astype(int), title="ZTF DR24 r-band — binary coverage mask")
plt.show()


## 2. Stripe 82 Footprint

Stripe 82 is the equatorial stripe at |Dec| < 1.5° spanning RA 310°–60°
(wrapping through 0°). We paint it onto the HEALPix grid as a binary mask.


In [ ]:
stripe82_mask = (
    (dec_grid > -1.5) & (dec_grid < 1.5) &
    ((ra_grid > 310) | (ra_grid < 60))
)

stripe82_map = np.zeros(NPIX)
stripe82_map[stripe82_mask] = 1

hp.mollview(stripe82_map, title="Stripe 82 footprint", min=0, max=1)
plt.show()


## 3. SDSS DR16Q Quasar Density Map

We convert quasar RA/Dec coordinates to HEALPix pixel indices and bin
them into a density map, then smooth slightly for visualisation.


In [ ]:
hdul_q   = fits.open("data/raw/DR16Q_v4.fits")
qso_data = hdul_q[1].data

ra_qso  = qso_data['RA']
dec_qso = qso_data['DEC']

# RA/Dec → HEALPix pixel
theta_q = np.radians(90.0 - dec_qso)
phi_q   = np.radians(ra_qso)
pix_qso = hp.ang2pix(NSIDE, theta_q, phi_q)

# Count quasars per pixel, then smooth for display
qso_density = np.bincount(pix_qso, minlength=NPIX).astype(float)
qso_density_smooth = hp.smoothing(qso_density, sigma=np.radians(0.3))

hp.mollview(qso_density_smooth, title="SDSS DR16Q quasar density (smoothed)")
plt.show()


## 4. Combined Coverage Map

We overlay all three footprints into a single categorical map so we can
immediately see where ZTF, DR16Q, and Stripe 82 overlap.

| Value | Meaning                        |
|-------|--------------------------------|
| 0     | No coverage                    |
| 1     | ZTF only                       |
| 2     | Stripe 82 only                 |
| 3     | DR16Q quasars only             |
| 4     | ZTF + Stripe 82                |
| 5     | ZTF + DR16Q                    |
| 6     | Stripe 82 + DR16Q              |
| 7     | All three overlap              |


In [ ]:
# ── Smooth and threshold ZTF for cleaner display ─────────────────────────────
ztf_smooth = hp.smoothing(ztf_map, sigma=np.radians(0.2))
ztf_mask   = ztf_smooth > 20          # >20 epochs = meaningful coverage

# ── Expand quasar footprint slightly for visual clarity ───────────────────────
qso_mask = np.zeros(NPIX, dtype=bool)
qso_mask[pix_qso] = True
qso_mask = hp.smoothing(qso_mask.astype(float), sigma=np.radians(0.3)) > 0

# ── Build categorical combined map ────────────────────────────────────────────
combined = np.zeros(NPIX)
combined[ztf_mask]                              = 1   # ZTF only
combined[stripe82_mask]                          = 2   # Stripe 82
combined[qso_mask]                               = 3   # DR16Q
combined[ztf_mask & stripe82_mask]               = 4
combined[ztf_mask & qso_mask]                    = 5
combined[stripe82_mask & qso_mask]               = 6
combined[ztf_mask & stripe82_mask & qso_mask]    = 7

hp.mollview(
    combined,
    title="ZTF DR24 + SDSS DR16Q + Stripe 82 — combined coverage",
    cmap="tab10",
    min=0, max=7,
)
plt.show()
